# Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import random
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import load_img,img_to_array
from tensorflow.keras.utils import to_categorical
plt.rcParams["font.size"]=15

# Data Loading

In [ ]:
base_dir="input"
class_labels=os.listdir(base_dir)
class_labels.sort()
print(class_labels)

In [ ]:
wdir_total = 0
data_classes_size=[]
data_labels=[]
for label in class_labels:
    total = len(os.listdir(os.path.join(base_dir, label)))
    print(f'Target name: {label}, number of samples: {total}')
    data_labels.append(label)
    data_classes_size.append(total)

In [ ]:
with plt.style.context('fivethirtyeight'):
    plt.figure(figsize=(10,5))
    plt.barh(y=data_labels,width=data_classes_size)
    plt.title(label="Size comparision of classes")
    plt.xlabel(xlabel='number of samples')
    plt.ylabel(ylabel='class labels')
    plt.show()

In [ ]:
with plt.style.context('fivethirtyeight'):
    plt.figure(figsize=(5,5))
    plt.pie(x=data_classes_size,
            labels=data_labels,
            autopct="%.2f%%",
            shadow=True,
           pctdistance=0.85)
    centre_circle=plt.Circle((0,0),0.70,fc='white')
    fig=plt.gcf()
    fig.gca().add_artist(centre_circle)
    plt.title(label="Size comparision of train-set classes")
    plt.show()

In [ ]:
data=[]
count=0

for label in class_labels:
    path = os.path.join(base_dir, label)
    print(path)
    images_list=list(os.listdir(path))
    for img in images_list:
        try:
            image=load_img(os.path.join(path, img), grayscale=False, color_mode='rgb',target_size=(224,224))
            image=img_to_array(image)
            image=image/255.0
            data.append([image,count])
        except Exception as e:
            continue
    count=count+1

In [ ]:
X,y =zip(*data)

In [ ]:
X[0]

In [ ]:
print(y)

In [ ]:
X=np.array(X)
y=np.array(y)

In [ ]:
from random import randint
plt.rcParams['font.size']=10
def show_images(X,y):
    labels =class_labels
    x,y = X,y
    plt.figure(figsize=(10, 10))
    for i in range(16):
        ax = plt.subplot(4, 4, i + 1)
        idx = randint(0,x.shape[0]-1)
        plt.imshow(x[idx])
        plt.axis("off")
        plt.title("Class:{}".format(labels[y[idx]]))
show_images(X,y)

# Data Splitting

In [ ]:
y=to_categorical(y)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)

In [ ]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

# Algorithm: ConvolutionalNeuralNetwork

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D,MaxPool2D,BatchNormalization,Dense,Dropout,Flatten

In [ ]:
cnn_model=Sequential()

cnn_model.add(Conv2D(filters=64,kernel_size=3,strides=(1,1),padding="same",activation="relu",input_shape = (224,224,3)))
cnn_model.add(Conv2D(filters=64,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPool2D(pool_size=(2,2)))

cnn_model.add(Conv2D(filters=128,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(Conv2D(filters=128,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPool2D(pool_size=(2,2),padding="same"))

cnn_model.add(Conv2D(filters=256,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(Conv2D(filters=256,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPool2D(pool_size=(2,2),padding="same"))

cnn_model.add(Conv2D(filters=512,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(Conv2D(filters=512,kernel_size=3,strides=(1,1),padding="same",activation="relu"))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPool2D(pool_size=(2,2),padding="same"))

cnn_model.add(Flatten())
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(units=512,activation="relu"))
cnn_model.add(Dense(units=len(class_labels),activation="softmax"))

In [ ]:
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
cnn_model.summary()

In [ ]:
history=cnn_model.fit(x=X,y=y,batch_size=32,epochs=20,validation_data=(X_train,y_train),shuffle=True)

In [ ]:
with plt.style.context(style='fivethirtyeight'):
    fig,axes=plt.subplots(nrows=1,ncols=2,figsize=(18,8))
    axes[0].plot(history.history["accuracy"],label="accuracy")
    axes[0].plot(history.history["val_accuracy"],label="val_accuracy")
    axes[0].set_title(label='Accuracy plot-graphs')
    axes[0].set_xlabel(xlabel='Number of epochs')
    axes[0].set_ylabel(ylabel='Accuracy')
    axes[0].legend()
    axes[1].plot(history.history["loss"],label="loss")
    axes[1].plot(history.history["val_loss"],label="val_loss")
    axes[1].set_title(label='Loss plot-graphs')
    axes[1].set_xlabel(xlabel='Number of epochs')
    axes[1].set_ylabel(ylabel='Loss')
    axes[1].legend()
    fig.tight_layout()
    plt.show()

In [ ]:
cnn_prediction=cnn_model.predict(X_test,batch_size=32,verbose=1)

In [ ]:
print(cnn_prediction)

In [ ]:
cnn_labels=[]
for i in range(len(cnn_prediction)):
    cnn_labels.append(np.argmax(cnn_prediction[i]))

In [ ]:
print(cnn_labels)

In [ ]:
true_labels=[]
for i in range(len(y_test)):
    true_labels.append(np.argmax(y_test[i]))

In [ ]:
print(true_labels)

# Result Analysis

***Accuracy Score***

In [ ]:
cnn_accuracy=accuracy_score(y_true=true_labels,y_pred=cnn_labels)
print(f'Validation accuracy of ComvolutionalNeuralNetwork model is {cnn_accuracy*100.0:.2f}%')

***Classification Report***

In [ ]:
print(classification_report(y_true=true_labels,y_pred=cnn_labels,target_names=class_labels))

***Confusion Matrix***

In [ ]:
plt.figure(figsize=(5,5))
sns.heatmap(data=confusion_matrix(y_true=true_labels,y_pred=cnn_labels),
            annot=True,
            fmt='4d',
            cbar=False,
            cmap=plt.cm.Greens,
            xticklabels=class_labels,
            yticklabels=class_labels)
plt.title(label='Confusion Matrix')
plt.show()

***Model Saving***

In [ ]:
cnn_model.save(filepath="model/ConvolutionalNeuralNetwork_model.h5")

## Thank You